# 🍽️ Exploration des Données Food.com

## Objectif du Notebook
Ce notebook permet d'explorer et d'analyser les données de recettes et d'interactions utilisateurs du dataset Food.com de Kaggle.

### Dataset Source
- **Lien Kaggle**: [Food.com Recipes and User Interactions](https://www.kaggle.com/datasets/shuyangli94/food-com-recipes-and-user-interactions)
- **Fichiers disponibles**:
  - `interactions_test.csv`, `interactions_train.csv`, `interactions_validation.csv`
  - `PP_recipes.csv`, `PP_users.csv`
  - `RAW_interactions.csv`, `RAW_recipes.csv`

### Plan d'analyse
1. **Configuration** - Structure du projet et imports
2. **Chargement** - Lecture de tous les fichiers CSV
3. **Vue d'ensemble** - Statistiques descriptives
4. **Relations** - Connexions entre les datasets
5. **Qualité** - Évaluation de la propreté des données
6. **Visualisation** - Graphiques exploratoires

## 1. 📁 Configuration et Structure du Projet

In [ ]:
# Imports des bibliothèques essentielles
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from pathlib import Path
import os

# Configuration des graphiques
plt.style.use('default')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)
warnings.filterwarnings('ignore')

# Configuration du répertoire de travail
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_DIR = PROJECT_ROOT / 'data'
RAW_DATA_DIR = DATA_DIR / 'raw'
PROCESSED_DATA_DIR = DATA_DIR / 'processed'

print(f"📂 Répertoire du projet: {PROJECT_ROOT}")
print(f"📂 Répertoire des données brutes: {RAW_DATA_DIR}")
print(f"📂 Répertoire des données traitées: {PROCESSED_DATA_DIR}")

# Vérification de l'existence des répertoires
for directory in [DATA_DIR, RAW_DATA_DIR, PROCESSED_DATA_DIR]:
    directory.mkdir(exist_ok=True, parents=True)
    print(f"✅ {directory.name.upper()}: {'Existe' if directory.exists() else 'Créé'}")

## 2. 📊 Chargement et Exploration des Fichiers CSV

**Instructions importantes:**
1. Placez tous vos fichiers CSV dans le dossier `data/raw/`
2. Les fichiers attendus sont:
   - `interactions_test.csv`, `interactions_train.csv`, `interactions_validation.csv`
   - `PP_recipes.csv`, `PP_users.csv`
   - `RAW_interactions.csv`, `RAW_recipes.csv`

In [ ]:
# Définition des fichiers de données
FILES = {
    'interactions_train': 'interactions_train.csv',
    'interactions_test': 'interactions_test.csv', 
    'interactions_validation': 'interactions_validation.csv',
    'PP_recipes': 'PP_recipes.csv',
    'PP_users': 'PP_users.csv',
    'RAW_interactions': 'RAW_interactions.csv',
    'RAW_recipes': 'RAW_recipes.csv'
}

# Vérification de l'existence des fichiers
print("📋 Vérification des fichiers:")
existing_files = {}
for key, filename in FILES.items():
    file_path = RAW_DATA_DIR / filename
    if file_path.exists():
        print(f"✅ {filename}")
        existing_files[key] = file_path
    else:
        print(f"❌ {filename} - MANQUANT")

print(f"\n🔍 Fichiers trouvés: {len(existing_files)}/{len(FILES)}")

if not existing_files:
    print("\n⚠️  Aucun fichier trouvé!")
    print("📁 Placez vos fichiers CSV dans:", RAW_DATA_DIR)

In [ ]:
# Fonction pour charger les données de manière sécurisée
def load_data_safe(file_path, max_rows=None):
    """Charge les données avec gestion d'erreurs"""
    try:
        if max_rows:
            df = pd.read_csv(file_path, nrows=max_rows)
            print(f"📊 Échantillon chargé ({max_rows} lignes)")
        else:
            df = pd.read_csv(file_path)
            print(f"📊 Dataset complet chargé")
        return df
    except Exception as e:
        print(f"❌ Erreur lors du chargement: {e}")
        return None

# Chargement des données disponibles
datasets = {}
for key, file_path in existing_files.items():
    print(f"\n📂 Chargement de {key}...")
    
    # Pour les gros fichiers, on charge d'abord un échantillon
    if 'RAW' in key:
        df = load_data_safe(file_path, max_rows=10000)  # Échantillon de 10k lignes
    else:
        df = load_data_safe(file_path)
    
    if df is not None:
        datasets[key] = df
        print(f"   📏 Forme: {df.shape}")
        print(f"   📋 Colonnes: {list(df.columns)}")

print(f"\n🎯 Datasets chargés avec succès: {len(datasets)}")

## 3. 🔍 Vue d'ensemble et Statistiques Descriptives

In [ ]:
# Analyse détaillée de chaque dataset
def analyze_dataset(name, df):
    """Analyse complète d'un dataset"""
    print(f"\n{'='*50}")
    print(f"📊 ANALYSE DE: {name.upper()}")
    print(f"{'='*50}")
    
    print(f"📏 Forme: {df.shape[0]:,} lignes × {df.shape[1]} colonnes")
    
    print("\n📋 Types de données:")
    print(df.dtypes.value_counts())
    
    print("\n🔍 Valeurs manquantes:")
    missing = df.isnull().sum()
    missing_pct = (missing / len(df)) * 100
    missing_info = pd.DataFrame({
        'Colonnes': df.columns,
        'Manquantes': missing,
        'Pourcentage': missing_pct.round(2)
    })
    print(missing_info[missing_info['Manquantes'] > 0].sort_values('Manquantes', ascending=False))
    
    print("\n📈 Statistiques numériques:")
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    if len(numeric_cols) > 0:
        print(df[numeric_cols].describe())
    
    print("\n🔤 Variables catégorielles:")
    categorical_cols = df.select_dtypes(include=['object', 'category']).columns
    for col in categorical_cols[:3]:  # Limiter à 3 colonnes pour éviter trop de texte
        unique_count = df[col].nunique()
        print(f"   {col}: {unique_count:,} valeurs uniques")
        if unique_count <= 10:
            print(f"      Valeurs: {df[col].value_counts().head().to_dict()}")

# Analyse de tous les datasets disponibles
for name, df in datasets.items():
    analyze_dataset(name, df)

## 4. 🔗 Relations entre les Datasets

In [ ]:
# Exploration des relations entre datasets
def explore_relationships():
    """Analyse les clés de jointure potentielles"""
    
    print("🔗 ANALYSE DES RELATIONS ENTRE DATASETS")
    print("="*50)
    
    # Recherche des colonnes communes
    all_columns = {}
    for name, df in datasets.items():
        all_columns[name] = set(df.columns)
    
    print("\n📊 Colonnes communes:")
    for i, (name1, cols1) in enumerate(all_columns.items()):
        for name2, cols2 in list(all_columns.items())[i+1:]:
            common = cols1.intersection(cols2)
            if common:
                print(f"   {name1} ↔ {name2}: {common}")
    
    # Analyse des clés potentielles
    potential_keys = ['id', 'user_id', 'recipe_id', 'contributor_id']
    
    print("\n🔑 Analyse des clés potentielles:")
    for key in potential_keys:
        print(f"\n   Clé: {key}")
        for name, df in datasets.items():
            if key in df.columns:
                unique_count = df[key].nunique()
                total_count = len(df)
                uniqueness = unique_count / total_count
                print(f"      {name}: {unique_count:,}/{total_count:,} ({uniqueness:.1%} unique)")

# Exécution si des données sont disponibles
if datasets:
    explore_relationships()
else:
    print("⚠️ Aucun dataset disponible pour l'analyse des relations")

## 5. 🧹 Évaluation de la Qualité des Données

In [ ]:
# Évaluation complète de la qualité des données
def data_quality_assessment(name, df):
    """Évaluation complète de la qualité d'un dataset"""
    
    print(f"\n🔍 QUALITÉ DES DONNÉES: {name.upper()}")
    print("-" * 40)
    
    # 1. Doublons
    duplicates = df.duplicated().sum()
    print(f"🔄 Doublons: {duplicates:,} ({duplicates/len(df):.1%})")
    
    # 2. Valeurs manquantes
    missing_total = df.isnull().sum().sum()
    missing_pct = missing_total / (len(df) * len(df.columns))
    print(f"❓ Valeurs manquantes: {missing_total:,} ({missing_pct:.1%})")
    
    # 3. Colonnes avec beaucoup de valeurs uniques (potentiellement problématiques)
    high_cardinality_cols = []
    for col in df.columns:
        if df[col].nunique() > len(df) * 0.9:  # Plus de 90% de valeurs uniques
            high_cardinality_cols.append(col)
    
    if high_cardinality_cols:
        print(f"🎯 Colonnes haute cardinalité: {high_cardinality_cols}")
    
    # 4. Valeurs aberrantes pour les colonnes numériques
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    if len(numeric_cols) > 0:
        print(f"📊 Colonnes numériques analysées: {len(numeric_cols)}")
        
        for col in numeric_cols[:3]:  # Limiter à 3 colonnes
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            outliers = ((df[col] < (Q1 - 1.5 * IQR)) | (df[col] > (Q3 + 1.5 * IQR))).sum()
            if outliers > 0:
                print(f"   {col}: {outliers} valeurs aberrantes potentielles")

# Analyse de qualité pour tous les datasets
for name, df in datasets.items():
    data_quality_assessment(name, df)

## 6. 📈 Visualisations Exploratoires Initiales

In [ ]:
# Créer des visualisations exploratoires
def create_initial_visualizations():
    """Génère des graphiques exploratoires selon les données disponibles"""
    
    if not datasets:
        print("⚠️ Aucune donnée disponible pour la visualisation")
        return
    
    # Configuration pour les graphiques
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle('🍽️ Visualisations Exploratoires - Food.com Dataset', fontsize=16, fontweight='bold')
    
    plot_count = 0
    
    for name, df in datasets.items():
        if plot_count >= 4:  # Limiter à 4 graphiques
            break
            
        ax = axes[plot_count//2, plot_count%2]
        
        # Choisir le type de graphique selon les données disponibles
        numeric_cols = df.select_dtypes(include=[np.number]).columns
        
        if len(numeric_cols) > 0:
            # Histogramme de la première colonne numérique
            col = numeric_cols[0]
            df[col].hist(bins=30, ax=ax, alpha=0.7, color='skyblue')
            ax.set_title(f'{name}: Distribution de {col}')
            ax.set_xlabel(col)
            ax.set_ylabel('Fréquence')
        else:
            # Graphique en barres pour les données catégorielles
            categorical_cols = df.select_dtypes(include=['object']).columns
            if len(categorical_cols) > 0:
                col = categorical_cols[0]
                top_values = df[col].value_counts().head(10)
                top_values.plot(kind='bar', ax=ax, color='lightcoral')
                ax.set_title(f'{name}: Top 10 {col}')
                ax.tick_params(axis='x', rotation=45)
            else:
                ax.text(0.5, 0.5, f'Pas de données\nvisualiser pour\n{name}', 
                       ha='center', va='center', transform=ax.transAxes)
                ax.set_title(f'{name}: Données non visualisables')
        
        plot_count += 1
    
    # Masquer les sous-graphiques non utilisés
    for i in range(plot_count, 4):
        axes[i//2, i%2].set_visible(False)
    
    plt.tight_layout()
    plt.show()

# Graphique de synthèse des datasets
def datasets_summary_plot():
    """Graphique de synthèse de tous les datasets"""
    
    if not datasets:
        print("⚠️ Aucune donnée disponible")
        return
    
    # Préparation des données pour le graphique de synthèse
    summary_data = []
    for name, df in datasets.items():
        summary_data.append({
            'Dataset': name,
            'Lignes': len(df),
            'Colonnes': len(df.columns),
            'Valeurs_manquantes': df.isnull().sum().sum()
        })
    
    summary_df = pd.DataFrame(summary_data)
    
    # Création du graphique
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
    
    # Nombre de lignes par dataset
    summary_df.plot(x='Dataset', y='Lignes', kind='bar', ax=ax1, color='lightgreen')
    ax1.set_title('📊 Taille des Datasets (Nombre de lignes)')
    ax1.tick_params(axis='x', rotation=45)
    ax1.set_ylabel('Nombre de lignes')
    
    # Valeurs manquantes par dataset
    summary_df.plot(x='Dataset', y='Valeurs_manquantes', kind='bar', ax=ax2, color='orange')
    ax2.set_title('❓ Valeurs Manquantes par Dataset')
    ax2.tick_params(axis='x', rotation=45)
    ax2.set_ylabel('Nombre de valeurs manquantes')
    
    plt.tight_layout()
    plt.show()
    
    return summary_df

# Génération des visualisations
print("📊 Génération des visualisations exploratoires...")
create_initial_visualizations()

print("\n📈 Synthèse des datasets...")
if datasets:
    summary = datasets_summary_plot()
    print("\n📋 Résumé numérique:")
    print(summary)

## 🎯 Prochaines Étapes

### Actions Immédiates
1. **📁 Placez vos fichiers CSV** dans le dossier `data/raw/`
2. **▶️ Exécutez ce notebook** pour explorer vos données
3. **🔍 Analysez les résultats** pour comprendre votre dataset

### Analyse Approfondie Suggérée
- **Nettoyage des données** (valeurs manquantes, doublons)
- **Analyse des patterns d'interaction** utilisateur-recette
- **Exploration des caractéristiques nutritionnelles**
- **Modélisation prédictive** (temps de préparation, recommandations)

### Notebooks Suivants à Créer
- `02_data_cleaning.ipynb` - Nettoyage et préparation
- `03_eda_advanced.ipynb` - Analyse exploratoire avancée
- `04_feature_engineering.ipynb` - Ingénierie des caractéristiques
- `05_modeling.ipynb` - Développement des modèles

---

**💡 Conseil**: Commencez par placer vos fichiers CSV dans `data/raw/` puis exécutez toutes les cellules de ce notebook pour avoir une vue d'ensemble de vos données !